# Customer Support Ticket Classification
End-to-end, reproducible NLP workflow. **Dataset note:** this repository currently uses explicitly synthetic development data because the source download was unavailable in the build environment.

## 1. Imports and setup

In [ ]:
from pathlib import Path
import sys, json
ROOT = Path.cwd().resolve()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import classification_report, ConfusionMatrixDisplay
from src.data_processing import load_and_clean

## 2. Dataset loading and inspection

In [ ]:
data_path = ROOT / 'data/raw/synthetic_support_tickets.csv'
df, cleaning = load_and_clean(data_path)
print('Shape:', df.shape, '| Cleaning:', cleaning)
df.head(3)

## 3. Data quality checks

In [ ]:
quality = pd.DataFrame({'dtype': df.dtypes.astype(str), 'missing': df.isna().sum()})
display(quality)
print('Duplicates:', df.duplicated().sum())

## 4. Exploratory analysis and visualizations

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
df.category.value_counts().plot.barh(ax=axes[0,0], title='Category distribution')
df.priority.value_counts().plot.bar(ax=axes[0,1], title='Priority distribution')
df.channel.value_counts().plot.bar(ax=axes[1,0], title='Channel distribution')
df.text.str.len().plot.hist(bins=25, ax=axes[1,1], title='Text length')
plt.tight_layout(); plt.show()
pd.crosstab(df.category, df.priority, normalize='index').round(2)

## 5. Feature preparation and stratified split
Subject and description are combined before splitting. TF-IDF is fitted only within each training pipeline, preventing test leakage.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df.text, df.category, test_size=.2, random_state=42, stratify=df.category)
def make(model): return Pipeline([('tfidf', TfidfVectorizer(stop_words='english', ngram_range=(1,2), min_df=2, max_df=.98)), ('classifier', model)])

## 6. Model training and comparison

In [ ]:
models = {'Logistic Regression': LogisticRegression(max_iter=1200, class_weight='balanced', random_state=42), 'LinearSVC': LinearSVC(class_weight='balanced', random_state=42), 'Multinomial NB': MultinomialNB(alpha=.3)}
rows, fitted = [], {}
for name, estimator in models.items():
    fitted[name] = make(estimator).fit(X_train, y_train)
    report = classification_report(y_test, fitted[name].predict(X_test), output_dict=True, zero_division=0)
    rows.append({'model': name, 'accuracy': report['accuracy'], 'macro_f1': report['macro avg']['f1-score'], 'weighted_f1': report['weighted avg']['f1-score']})
comparison = pd.DataFrame(rows).sort_values('macro_f1', ascending=False)
comparison

## 7. Evaluation and confusion matrix

In [ ]:
best_name = comparison.iloc[0].model
best = fitted[best_name]
predictions = best.predict(X_test)
print(classification_report(y_test, predictions, zero_division=0))
ConfusionMatrixDisplay.from_predictions(y_test, predictions, xticks_rotation=35, cmap='Blues', colorbar=False); plt.tight_layout(); plt.show()

## 8. Explainability: influential terms

In [ ]:
lr = fitted['Logistic Regression']; terms = lr.named_steps['tfidf'].get_feature_names_out(); clf = lr.named_steps['classifier']
for label, weights in zip(clf.classes_, clf.coef_): print(label, ':', ', '.join(terms[weights.argsort()[-8:][::-1]]))

## 9. Error analysis

In [ ]:
errors = pd.DataFrame({'text': X_test, 'actual': y_test, 'predicted': predictions})
errors[errors.actual != errors.predicted].head(10)

## 10. Sample predictions

In [ ]:
samples = ['I reset my password but my account is still locked', 'The tracking page says my parcel went to the wrong address']
list(zip(samples, best.predict(samples)))

## 11. Key findings, limitations, and conclusion
All three classical models separate the synthetic templates perfectly. This is a pipeline verification result—not evidence of real-world generalization. The balanced, templated classes are much easier than production traffic. Replace the fallback with the documented CC0 Kaggle data, review label quality, retrain, and re-evaluate before making operational claims. The project nevertheless validates the complete reproducible path from cleaning through serving.